In [ ]:
import os
import sys
import subprocess
try:
   import pyarrow
except ImportError:
   subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "-q"])
os.environ["JAVA_HOME"] = r"D:\java\openjdk-8u482-b08"
os.environ["HADOOP_HOME"] = r"D:\java\hadoop-3.4.3"
os.environ["SPARK_HOME"] = r"D:\BIGDATA_G16\.venv\Lib\site-packages\pyspark"
os.environ["SPARK_LOCAL_DIRS"] = r"D:\BIGDATA_G16\Spark_Temp"
sys.path.append(r"D:\java\hadoop-3.4.3\bin")
sys.path.append(r"D:\BIGDATA_G16\.venv\Scripts")
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType, TimestampType
try:
   SparkContext.getOrCreate().stop()
except Exception:
   pass
spark = SparkSession.builder \
   .appName("MetroPT3_MasterNode_Group16_EDA") \
   .master("local[*]") \
   .config("spark.executor.memory", "2g") \
   .config("spark.driver.memory", "2g") \
   .config("spark.local.dir", r"D:\BIGDATA_G16\Spark_Temp") \
   .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
   .getOrCreate()
hdfs_path = "hdfs://10.125.222.18:9000/Group16/MetroPT3(AirCompressor).csv"
df_raw = spark.read.csv(hdfs_path, header=True, inferSchema=False)
analog_cols = ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current']
digital_cols = ['COMP', 'DV_electric', 'DV_eletric', 'TOWERS', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
df = df_raw
for c in df.columns:
   c_upper = c.upper()
   if c_upper in [x.upper() for x in analog_cols]:
       df = df.withColumn(c, col(c).cast(DoubleType()))
   elif c_upper in [x.upper() for x in digital_cols]:
       df = df.withColumn(c, col(c).cast(IntegerType()))
   elif c_upper == "TIMESTAMP":
       df = df.withColumn(c, col(c).cast(TimestampType()))
df_clean = df
df.printSchema()
print("----------------------------------------------------------------")
print(f"-> Tổng số dòng dữ liệu: {df.count():,} dòng.")
print("----------------------------------------------------------------")

In [ ]:
print("--- KIỂM TRA SCHEMA CỦA TẬP DỮ LIỆU SƠ CẤP ---")
df.printSchema()

In [ ]:
analog_cols = ["TP2", "TP3", "H1", "DV_pressure", "Reservoirs", "Oil_temperature", "Motor_current"]
raw_summary = df.select(analog_cols).summary("min", "max", "mean", "stddev", "25%", "50%", "75%").toPandas()
summary_analog_df = raw_summary.set_index("summary").transpose().reset_index()
summary_analog_df.columns = ["Feature", "Min", "Max", "Mean", "Stddev", "Q1", "Median", "Q3"]
print("\n--- BẢNG KẾT QUẢ THỐNG KÊ MÔ TẢ ĐẦY ĐỦ CÁC CẢM BIẾN ANALOG ---")
summary_analog_df

In [ ]:
from pyspark.sql.functions import col, count, when, round
digital_cols = ["COMP", "DV_eletric", "Towers", "MPG", "LPS", "Pressure_switch", "Oil_level", "Caudal_impulses"]
total_count = df.count()
exprs = []
for c in digital_cols:
    display_name = "DV_electric" if c == "DV_eletric" else c
    exprs.append(round((count(when(col(c) == '0.0', 1)) / total_count) * 100, 1).alias(f"{display_name}_%_0"))
    exprs.append(round((count(when(col(c) == '1.0', 1)) / total_count) * 100, 1).alias(f"{display_name}_%_1"))
summary_raw = df.select(exprs).collect()[0].asDict()
rows = []
for c in digital_cols:
    display_name = "DV_electric" if c == "DV_eletric" else c
    p0 = f"{summary_raw[f'{display_name}_%_0']}%"
    p1 = f"{summary_raw[f'{display_name}_%_1']}%"
    rows.append({
        "Tín hiệu": display_name,
        "% = 0": p0,
        "% = 1": p1
    })
import pandas as pd
summary_digital_df = pd.DataFrame(rows)
print("\n--- BẢNG KẾT QUẢ THỐNG KÊ TẦN SUẤT TÍN HIỆU DIGITAL TỪ HDFS ---")
summary_digital_df

In [ ]:
from pyspark.sql.functions import col, count, when, isnan
import pandas as pd
null_exprs = []
for c in df.columns:
    if c == "timestamp":
        null_exprs.append(count(when(col(c).isNull(), 1)).alias(c))
    else:
        null_exprs.append(count(when(col(c).isNull() | isnan(col(c).cast("double")), 1)).alias(c))
null_counts_dict = df.select(null_exprs).collect()[0].asDict()
missing_rows = []
for col_name, missing_count in null_counts_dict.items():
    display_name = "DV_electric" if col_name == "DV_eletric" else col_name
    percentage = (missing_count / 1516948) * 100

    missing_rows.append({
        "Thuộc tính": display_name,
        "Số dòng khuyết thiếu (Null/NaN)": missing_count,
        "Tỷ lệ (%)": f"{percentage:.3f}%",
    })
df_missing_summary = pd.DataFrame(missing_rows)

print("\n--- BẢNG KẾT QUẢ-")
df_missing_summary

In [ ]:
total_raw_rows = df.count()
distinct_raw_rows = df.dropDuplicates().count()
duplicate_exist = total_raw_rows - distinct_raw_rows

print(f"-> Tổng số lượng bản ghi hiện có trong tập dữ liệu gốc: {total_raw_rows:,} dòng.")
print(f"-> Số lượng bản ghi định danh duy nhất (Distinct):     {distinct_raw_rows:,} dòng.")
print(f"-> Số lượng bản ghi trùng lặp tồn tại trong dữ liệu gốc: {duplicate_exist} dòng.")

In [ ]:
from pyspark.sql.functions import col
import pandas as pd
import builtins

# 1. Định nghĩa hàm quét phân tán để đếm số lượng
def get_raw_outlier_pct(df_spark, column_name, q1, q3):
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Bộ lọc đếm số dòng nằm ngoài biên độ IQR của dữ liệu gốc
    outliers_count = df_spark.filter((col(column_name).cast("double") < lower_bound) |
                                     (col(column_name).cast("double") > upper_bound)).count()
    outlier_pct = (outliers_count / 1516948) * 100
    return builtins.round(outlier_pct, 1)

# 2. Khai báo các thông số Q1, Q3 thu thập từ ma trận thống kê mô tả sơ cấp
analog_specs = [
    {"Feature": "TP2", "Q1": -0.014, "Q3": -0.010, "Giải thích": "Trạng thái máy tắt / khởi động / Pre-Failure"},
    {"Feature": "TP3", "Q1": 8.492, "Q3": 9.492, "Giải thích": "Biến động áp suất panel khi chuyển trạng thái"},
    {"Feature": "H1", "Q1": 8.254, "Q3": 9.374, "Giải thích": "Đột biến khi bộ lọc cyclonic xả — sự kiện bình thường"},
    {"Feature": "DV_pressure", "Q1": -0.022, "Q3": -0.018, "Giải thích": "Tháp sấy xả ẩm — xảy ra định kỳ theo chu kỳ"},
    {"Feature": "Reservoirs", "Q1": 8.494, "Q3": 9.492, "Giải thích": "Áp suất bình tích khí chính biến động chu kỳ chạy tải"},
    {"Feature": "Oil_temperature", "Q1": 57.775, "Q3": 67.250, "Giải thích": "Máy vừa khởi động (thấp) hoặc quá nhiệt (cao)"},
    {"Feature": "Motor_current", "Q1": 0.040, "Q3": 3.808, "Giải thích": "Trạng thái tắt (~0A) hoặc vượt tải (>9A)"}
]

# 3. Tiến hành truy vấn song song và cấu trúc hóa bảng kết quả thống kê
outlier_rows = []
for spec in analog_specs:
    f_name = spec["Feature"]
    q1_val = spec["Q1"]
    q3_val = spec["Q3"]
    iqr_val = builtins.round(q3_val - q1_val, 3)

    pct_outlier = get_raw_outlier_pct(df, f_name, q1_val, q3_val)

    outlier_rows.append({
        "Feature": f_name,
        "Q1": q1_val,
        "Q3": q3_val,
        "IQR": iqr_val,
        "% Outlier gốc": f"~{pct_outlier}%",
        "Ý nghĩa kỹ thuật": spec["Giải thích"]
    })

df_outlier_summary = pd.DataFrame(outlier_rows)

print("\n--- BẢNG TỔNG HỢP ---")
df_outlier_summary

In [ ]:
# Import các hàm xử lý Spark cần thiết
from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek, avg, count
from pyspark.sql.types import IntegerType

# Định nghĩa danh sách thuộc tính dựa trên schema dữ liệu gốc của nhóm
analog_cols = ["TP2", "TP3", "H1", "DV_pressure", "Reservoirs", "Oil_temperature", "Motor_current"]
digital_cols = ["COMP", "DV_eletric", "Towers", "MPG", "LPS", "Pressure_switch", "Oil_level", "Caudal_impulses"]

In [ ]:
# =========================================================================
# Ô CODE TỔNG HỢP: TIỀN XỬ LÝ (3.4) VÀ THỐNG KÊ CHUỖI THỜI GIAN (3.5)
# =========================================================================

from pyspark.sql.functions import col, to_timestamp, hour, month, dayofweek, avg, count
from pyspark.sql.types import IntegerType
import pandas as pd
import builtins

# -------------------------------------------------------------------------
# MỤC 3.4: TIỀN XỬ LÝ DỮ LIỆU
# -------------------------------------------------------------------------
df_clean = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))
df_clean = df_clean.withColumnRenamed("DV_eletric", "DV_electric")

digital_cols_clean = ["COMP", "DV_electric", "Towers", "MPG", "LPS", "Pressure_switch", "Oil_level", "Caudal_impulses"]
for c in digital_cols_clean:
    df_clean = df_clean.withColumn(c, col(c).cast(IntegerType()))

df_clean = df_clean \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("month", month(col("timestamp"))) \
    .withColumn("dow", dayofweek(col("timestamp")))

In [ ]:
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df_clean.write.mode("overwrite").parquet(HDFS_CLEAN_FOR_SQL)